<a href="https://colab.research.google.com/github/villaalessandra03-svg/Project_NLP_villa/blob/main/sentence_embeddings_prova1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

df = pd.read_csv("jo_sentences_clustering.csv")

print("Numero di frasi:", len(df))
display(df.head())

Saving jo_sentences_clustering.csv to jo_sentences_clustering (1).csv
Numero di frasi: 1959


,novel,sentence_id,sentence,start_offset,end_offset
0,littlewomen,littlewomen_00001,“Christmas won’t be Christmas without any pres...,74,157
1,littlewomen,littlewomen_00002,The four young faces on which the firelight sh...,469,654
2,littlewomen,littlewomen_00003,"I’ve wanted it so long,” said Jo, who was a bo...",1515,1568
3,littlewomen,littlewomen_00004,"Let’s each buy what we want, and have a little...",1885,2046
4,littlewomen,littlewomen_00005,"“You don’t have half such a hard time as I do,...",2196,2252


In [3]:
%pip install -q sentence-transformers

In [4]:
import torch
from sentence_transformers import SentenceTransformer

# Verifica della GPU
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Dispositivo utilizzato:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Caricamento del modello
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Modello caricato correttamente!")
print("Dimensioni degli embeddings:", model.get_sentence_embedding_dimension())

Dispositivo utilizzato: cuda
GPU: Tesla T4


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modello caricato correttamente!
Dimensioni degli embeddings: 384


/tmp/ipykernel_749/3006011574.py:19: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Dimensioni degli embeddings:", model.get_sentence_embedding_dimension())


In [5]:
import numpy as np

# Recuperiamo le frasi dal dataset CLiC
sentences = df["sentence"].astype(str).tolist()

print("Numero di frasi:", len(sentences))

# Generazione degli embeddings
embeddings = model.encode(
    sentences,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# Controllo del risultato
print("\nEmbeddings generati correttamente!")
print("Dimensioni della matrice:", embeddings.shape)

# Salvataggio della matrice
np.save("jo_march_embeddings.npy", embeddings)

print("File salvato: jo_march_embeddings.npy")

Numero di frasi: 1959


Batches:   0%|          | 0/31 [00:00<?, ?it/s]


Embeddings generati correttamente!
Dimensioni della matrice: (1959, 384)
File salvato: jo_march_embeddings.npy


In [6]:
%pip install -q umap-learn hdbscan

In [7]:
import umap
import numpy as np

# Riduzione dimensionale degli embeddings
reducer = umap.UMAP(
    n_neighbors=15,
    n_components=10,
    min_dist=0.0,
    metric="cosine",
    random_state=42
)

embeddings_umap = reducer.fit_transform(embeddings)

print("Riduzione UMAP completata!")
print("Dimensioni originali:", embeddings.shape)
print("Dimensioni dopo UMAP:", embeddings_umap.shape)

# Salviamo anche la rappresentazione ridotta
np.save("jo_march_embeddings_umap10.npy", embeddings_umap)

print("File salvato: jo_march_embeddings_umap10.npy")

/usr/local/lib/python3.13/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Riduzione UMAP completata!
Dimensioni originali: (1959, 384)
Dimensioni dopo UMAP: (1959, 10)
File salvato: jo_march_embeddings_umap10.npy


In [8]:
import hdbscan
import numpy as np
import pandas as pd

# Clustering delle frasi su Jo March
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20,
    min_samples=5,
    metric="euclidean",
    cluster_selection_method="eom"
)

cluster_labels = clusterer.fit_predict(embeddings_umap)

# Aggiungiamo il cluster al dataset originale
df["cluster"] = cluster_labels

# Numero di cluster individuati (escluso il noise)
n_clusters = len(set(cluster_labels) - {-1})

# Frasi non assegnate
n_noise = np.sum(cluster_labels == -1)

print("Clustering HDBSCAN completato!")
print("Numero di cluster individuati:", n_clusters)
print("Frasi non assegnate (noise):", n_noise)
print("Percentuale di noise:", round(n_noise / len(df) * 100, 2), "%")

print("\nDistribuzione delle frasi per cluster:")
print(df["cluster"].value_counts().sort_index())

# Salvataggio dei risultati
df.to_csv("jo_march_clusters.csv", index=False)

print("\nFile salvato: jo_march_clusters.csv")

Clustering HDBSCAN completato!
Numero di cluster individuati: 20
Frasi non assegnate (noise): 663
Percentuale di noise: 33.84 %

Distribuzione delle frasi per cluster:
cluster
-1     663
 0     134
 1      32
 2      33
 3     133
 4      41
 5      69
 6      63
 7      49
 8      38
 9      47
 10    177
 11     68
 12     91
 13     89
 14     35
 15     23
 16     28
 17     97
 18     21
 19     28
Name: count, dtype: int64

File salvato: jo_march_clusters.csv


In [9]:
# Aggiungiamo il grado di appartenenza al cluster
df["cluster_probability"] = clusterer.probabilities_

# Mostriamo le 5 frasi con appartenenza più forte per ogni cluster
for cluster_id in sorted(df["cluster"].unique()):

    if cluster_id == -1:
        continue

    cluster_df = df[df["cluster"] == cluster_id]

    representative = cluster_df.sort_values(
        "cluster_probability",
        ascending=False
    ).head(5)

    print("\n" + "=" * 80)
    print(f"CLUSTER {cluster_id} — {len(cluster_df)} frasi")
    print("=" * 80)

    for _, row in representative.iterrows():
        print(f"\n[{row['novel']}]")
        print(row["sentence"])
        print(f"Probabilità: {row['cluster_probability']:.3f}")


CLUSTER 0 — 134 frasi

[littlewomen]
“I’m the oldest,” began Meg, but Jo cut in with a decided, “I’m the man of the family now Papa is away, and I shall provide the slippers, for he told me to take special care of Mother while he was gone.”
Probabilità: 1.000

[littlewomen]
They all drew to the fire, Mother in the big chair with Beth at her feet, Meg and Amy perched on either arm of the chair, and Jo leaning on the back, where no one would see any sign of emotion if the letter should happen to be touching.
Probabilità: 1.000

[littlewomen]
Jo quite glowed with pleasure at this boyish praise of her sister, and stored it up to repeat to Meg.
Probabilità: 1.000

[littlewomen]
So Meg went away to ‘accept with thanks’, look over her dress, and sing blithely as she did up her one real lace frill, while Jo finished her story, her four apples, and had a game of romps with Scrabble.
Probabilità: 1.000

[littlewomen]
After various lesser mishaps, Meg was finished at last, and by the united exer

In [10]:
# Numero di frasi per cluster e per romanzo
cluster_counts = pd.crosstab(
    df["cluster"],
    df["novel"]
)

# Percentuali rispetto al totale delle frasi di ciascun romanzo
cluster_percentages = pd.crosstab(
    df["cluster"],
    df["novel"],
    normalize="columns"
) * 100

print("FREQUENZE ASSOLUTE")
display(cluster_counts)

print("\nPERCENTUALI PER ROMANZO")
display(cluster_percentages.round(2))

FREQUENZE ASSOLUTE


novel,josboys,littlemen,littlewomen
cluster,,,
-1,173,127,363
0,12,5,117
1,4,14,14
2,6,12,15
3,9,5,119
4,0,0,41
5,0,1,68
6,1,0,62
7,4,2,43



PERCENTUALI PER ROMANZO


novel,josboys,littlemen,littlewomen
cluster,,,
-1,51.80,42.47,27.38
0,3.59,1.67,8.82
1,1.20,4.68,1.06
2,1.80,4.01,1.13
3,2.69,1.67,8.97
4,0.00,0.00,3.09
5,0.00,0.33,5.13
6,0.30,0.00,4.68
7,1.20,0.67,3.24
